In [2]:
import os, sys, json, random, time
import numpy as np
import pandas as pd
import igraph as ig
from pathlib import Path

# One seed, set everywhere, so this notebook reproduces exactly.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Paths are relative to notebooks/, so ROOT lands on the repo root.
ROOT = Path("..").resolve()
DATA_PROC = ROOT / "data/processed"
OUT_DIR = ROOT / "outputs"


print("igraph version:", ig.__version__)
print("Graph present:", (DATA_PROC / "04_graph_train.pkl").exists())

igraph version: 0.11.9
Graph present: True


In [3]:
df = pd.read_parquet(DATA_PROC / "01_audit.parquet")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
print(f"Loaded {len(df):,} transactions")

# 70/15/15 cut points, taken from the timestamp itself.
t70 = df["Timestamp"].quantile(0.70)
t85 = df["Timestamp"].quantile(0.85)

# Each row gets its label from its own timestamp, so row order cannot corrupt it.
df["split"] = np.where(df["Timestamp"] <= t70, "train",
              np.where(df["Timestamp"] <= t85, "val", "test"))

# Stop the notebook if the rebuild disagrees with the registered split by even one row.
counts = df["split"].value_counts().to_dict()
assert counts == {"train": 3554957, "val": 761749, "test": 761639}, f"Split mismatch: {counts}"
print("Split reproduced:", counts)

Loaded 5,078,345 transactions
Split reproduced: {'train': 3554957, 'val': 761749, 'test': 761639}


In [4]:
t0 = time.time()
g = ig.Graph.Read_Pickle(str(DATA_PROC / "04_graph_train.pkl"))
print(f"Loaded graph in {time.time() - t0:.1f}s")

# The graph must be exactly the one registered, or every feature below is off-plan.
assert g.vcount() == 513284, f"Vertex count mismatch: {g.vcount():,}"
assert g.ecount() == 3554957, f"Edge count mismatch: {g.ecount():,}"
print(f"Vertices: {g.vcount():,}  Edges: {g.ecount():,}  Directed: {g.is_directed()}")

# Self-loops are accounts paying themselves; they complicate "distinct counterparty".
n_loops = sum(g.is_loop())
print(f"Self-loops: {n_loops:,}")

# How often does the same pair transact repeatedly? This is the gap between the two degree definitions.
mult = np.array(g.count_multiple())
print(f"Edges sitting on a repeated pair: {(mult > 1).sum():,} ({(mult > 1).mean():.1%})")
print(f"Distinct ordered account pairs: {len(set(g.get_edgelist())):,}")

Loaded graph in 3.2s
Vertices: 513,284  Edges: 3,554,957  Directed: True
Self-loops: 558,821
Edges sitting on a repeated pair: 3,068,302 (86.3%)
Distinct ordered account pairs: 955,904


In [5]:
SENDER_COL, RECEIVER_COL = "Account", "Account.1"

# Cast to str so the comparison matches how Notebook 04 built the vertices.
train = df[df["split"] == "train"].copy()
train[SENDER_COL] = train[SENDER_COL].astype(str)
train[RECEIVER_COL] = train[RECEIVER_COL].astype(str)

# A self-loop is simply a row whose sender and receiver are the same account.
loop_mask = train[SENDER_COL] == train[RECEIVER_COL]
print(f"Self-loop rows: {loop_mask.sum():,} of {len(train):,} ({loop_mask.mean():.1%})")
print(f"Matches graph self-loop count: {loop_mask.sum() == 558821}")
print(f"Distinct accounts doing it: {train.loc[loop_mask, SENDER_COL].nunique():,}\n")

# Compare the two groups on the features that separate a conversion from a transfer.
for name, sub in [("SELF-LOOP", train[loop_mask]), ("OTHER", train[~loop_mask])]:
    same_ccy = (sub["Payment Currency"] == sub["Receiving Currency"]).mean()
    same_amt = np.isclose(sub["Amount Paid"], sub["Amount Received"]).mean()
    print(f"{name}: n={len(sub):,} | illicit={sub['Is Laundering'].mean():.4%} "
          f"| same currency={same_ccy:.1%} | paid==received={same_amt:.1%}")

# Payment format is the clearest signal of what kind of movement this is.
print("\nPayment format share, self-loops vs other:")
fmt = pd.DataFrame({
    "self_loop_%": train.loc[loop_mask, "Payment Format"].value_counts(normalize=True) * 100,
    "other_%":     train.loc[~loop_mask, "Payment Format"].value_counts(normalize=True) * 100,
}).round(1)
print(fmt.fillna(0))

# Twenty actual rows, so we are reading data rather than summary statistics.
cols = ["Timestamp", SENDER_COL, "Amount Paid", "Payment Currency",
        "Amount Received", "Receiving Currency", "Payment Format", "Is Laundering"]
print("\nSample of 20 self-loop rows:")
print(train.loc[loop_mask, cols].sample(20, random_state=SEED).to_string(index=False))

Self-loop rows: 558,821 of 3,554,957 (15.7%)
Matches graph self-loop count: True
Distinct accounts doing it: 367,282

SELF-LOOP: n=558,821 | illicit=0.0011% | same currency=91.7% | paid==received=91.7%
OTHER: n=2,996,136 | illicit=0.0951% | same currency=100.0% | paid==received=100.0%

Payment format share, self-loops vs other:
                self_loop_%  other_%
Payment Format                      
ACH                     8.5     11.8
Bitcoin                 4.0      2.7
Cash                    0.2     10.9
Cheque                  0.7     41.4
Credit Card             0.5     29.4
Reinvestment           86.1      0.0
Wire                    0.1      3.9

Sample of 20 self-loop rows:
          Timestamp   Account  Amount Paid Payment Currency  Amount Received Receiving Currency Payment Format  Is Laundering
2022-09-01 00:24:00 810351BA0        54.38           Shekel            54.38             Shekel   Reinvestment              0
2022-09-01 00:11:00 8075D2520      1693.63             

In [6]:
# Work on the whole dataset here, not just train, so the claim covers the benchmark.
full = df.copy()
full[SENDER_COL] = full[SENDER_COL].astype(str)
full[RECEIVER_COL] = full[RECEIVER_COL].astype(str)

full["is_self_loop"] = full[SENDER_COL] == full[RECEIVER_COL]
full["cross_ccy"] = full["Payment Currency"] != full["Receiving Currency"]

# The core question: of all cross-currency rows, what share are an account converting with itself?
xt = pd.crosstab(full["cross_ccy"], full["is_self_loop"])
print("Rows by cross-currency x self-loop:\n", xt, "\n")

n_cross = full["cross_ccy"].sum()
n_cross_loop = (full["cross_ccy"] & full["is_self_loop"]).sum()
print(f"Cross-currency rows: {n_cross:,} ({full['cross_ccy'].mean():.2%} of dataset)")
print(f"...of which self-loops: {n_cross_loop:,} ({n_cross_loop / n_cross:.2%})")

# If cross-currency movement almost never happens between two parties, say so precisely.
n_cross_between = n_cross - n_cross_loop
print(f"...cross-currency between DIFFERENT accounts: {n_cross_between:,}\n")

# Does any of it carry laundering signal?
print(f"Illicit among cross-currency rows: {full.loc[full['cross_ccy'], 'Is Laundering'].sum():,}")
print(f"Illicit among cross-currency self-loops: "
      f"{full.loc[full['cross_ccy'] & full['is_self_loop'], 'Is Laundering'].sum():,}")

# Which currencies the conversions run between, top pairs.
print("\nTop currency pairs among cross-currency rows:")
print(full.loc[full["cross_ccy"]]
        .groupby(["Payment Currency", "Receiving Currency"]).size()
        .sort_values(ascending=False).head(8).to_string())

Rows by cross-currency x self-loop:
 is_self_loop    False   True 
cross_ccy                    
False         4484942  521233
True             2191   69979 

Cross-currency rows: 72,170 (1.42% of dataset)
...of which self-loops: 69,979 (96.96%)
...cross-currency between DIFFERENT accounts: 2,191

Illicit among cross-currency rows: 0
Illicit among cross-currency self-loops: 0

Top currency pairs among cross-currency rows:
Payment Currency  Receiving Currency
US Dollar         Euro                  15838
Euro              US Dollar             11060
Yuan              US Dollar              6675
US Dollar         Yuan                   2547
                  Swiss Franc            2507
                  UK Pound               2489
                  Rupee                  2310
                  Shekel                 2204


In [7]:
import time, json
import igraph as ig
import numpy as np
import pandas as pd

# If cells 1-5 already loaded `g` and defined DATA_PROC, delete these two lines.
g = ig.Graph.Read_Pickle(str(DATA_PROC / "04_graph_train.pkl"))
print(f"loaded: {g.vcount():,} vertices, {g.ecount():,} edges, directed={g.is_directed()}")

timings = {}

# Simple projection: parallel edges collapsed, self-loops dropped. Degrees only.
t0 = time.perf_counter()
g_simple = g.copy()
del g_simple.es["weight"]              # a collapsed edge has no single weight
g_simple.simplify(multiple=True, loops=True)
timings["build_simple_projection"] = time.perf_counter() - t0

# Same collapse but KEEPING loops, so we can re-derive the 955,904 pair count.
g_pairs = g.copy()
del g_pairs.es["weight"]
g_pairs.simplify(multiple=True, loops=False)

print(f"distinct ordered pairs, self-pairs included: {g_pairs.ecount():,}   (expect 955,904)")
print(f"distinct ordered pairs, self-pairs excluded: {g_simple.ecount():,}")
print(f"difference: {g_pairs.ecount() - g_simple.ecount():,}   (expect 367,282)")
print(f"edges per pair: {g.ecount() / g_pairs.ecount():.3f}   (expect 3.719)")
del g_pairs

assert not any(g_simple.is_loop()), "self-loops survived the simplify"

# Integrity check (i): the account where the two definitions disagree most.
inc_in, dis_in = g.indegree(), g_simple.indegree()
gap = np.array(inc_in) - np.array(dis_in)
v = int(gap.argmax())
print(f"\nintegrity check (i) — account {g.vs[v]['name']}")
print(f"  incident-edge in-degree (transactions) : {inc_in[v]:,}")
print(f"  distinct-counterparty in-degree (D5)   : {dis_in[v]:,}")
print(f"  self-loop present: {g.are_connected(v, v)}")

loaded: 513,284 vertices, 3,554,957 edges, directed=True
distinct ordered pairs, self-pairs included: 955,904   (expect 955,904)
distinct ordered pairs, self-pairs excluded: 588,622
difference: 367,282   (expect 367,282)
edges per pair: 3.719   (expect 3.719)

integrity check (i) — account 8003231B0
  incident-edge in-degree (transactions) : 72
  distinct-counterparty in-degree (D5)   : 2
  self-loop present: True


C:\Users\chara\AppData\Local\Temp\ipykernel_59068\890719187.py:39: DeprecationWarning: Graph.are_connected() is deprecated; use Graph.are_adjacent() instead
  print(f"  self-loop present: {g.are_connected(v, v)}")


In [8]:
def timed(name, fn):
    t0 = time.perf_counter()
    out = fn()
    timings[name] = time.perf_counter() - t0
    print(f"{name}: {timings[name]:.1f}s")
    return out

# PageRank and strengths: registered on the directed multigraph, all edges, loops in.
pr           = timed("pagerank",    lambda: g.pagerank(damping=0.85, weights="weight", directed=True))
in_strength  = timed("in_strength", lambda: g.strength(mode="in",  weights="weight", loops=True))
out_strength = timed("out_strength",lambda: g.strength(mode="out", weights="weight", loops=True))

# Degrees: distinct counterparties, self-loops excluded (decision (a), 26 Aug).
in_deg  = timed("in_degree",  lambda: g_simple.indegree())
out_deg = timed("out_degree", lambda: g_simple.outdegree())

# Clustering: undirected simple projection; zero when degree < 2 (D8).
def _clustering():
    gu = g_simple.as_undirected(mode="collapse")
    return gu.transitivity_local_undirected(mode="zero")
clustering = timed("clustering", _clustering)

# Two-hop outward reach: |N2(v) \ (N1(v) u {v})|.
def _two_hop():
    succ = g_simple.get_adjlist(mode="out")
    out = []
    for i, one in enumerate(succ):
        if not one:
            out.append(0); continue
        two = set()
        for u in one:
            two.update(succ[u])
        two.difference_update(one); two.discard(i)
        out.append(len(two))
        if i % 100_000 == 0 and i:
            print(f"  two-hop {i:,}/{len(succ):,}")
    return out
two_hop = timed("two_hop", _two_hop)

features = pd.DataFrame({
    "account_id":    [str(n) for n in g.vs["name"]],
    "g_pagerank":    pr,
    "g_in_degree":   in_deg,
    "g_out_degree":  out_deg,
    "g_in_weighted": in_strength,
    "g_out_weighted":out_strength,
    "g_clustering":  clustering,
    "g_net_flow":    np.asarray(in_strength) - np.asarray(out_strength),
    "g_two_hop_out": two_hop,
})

assert len(features) == 513_284 and features["account_id"].is_unique
features.to_parquet(DATA_PROC / "05_vertex_features.parquet", compression="zstd")
with open(DATA_PROC / "05_feature_timings.json", "w") as f:
    json.dump(timings, f, indent=2)

print(f"\nsaved {len(features):,} accounts")
print(features[["g_in_degree", "g_out_degree", "g_pagerank"]].describe())

pagerank: 0.4s
in_strength: 0.1s
out_strength: 0.1s
in_degree: 0.0s
out_degree: 0.0s
clustering: 0.4s
  two-hop 100,000/513,284
  two-hop 300,000/513,284
  two-hop 500,000/513,284
two_hop: 1.8s

saved 513,284 accounts
         g_in_degree   g_out_degree    g_pagerank
count  513284.000000  513284.000000  5.132840e+05
mean        1.146776       1.146776  1.948239e-06
std         2.199222      24.230925  2.883664e-06
min         0.000000       0.000000  3.149489e-07
25%         0.000000       0.000000  3.428615e-07
50%         1.000000       1.000000  1.344655e-06
75%         2.000000       1.000000  2.099729e-06
max       545.000000   14230.000000  1.446087e-04


In [9]:
cols = ["g_clustering", "g_two_hop_out", "g_in_weighted", "g_out_weighted", "g_net_flow"]
print(features[cols].describe())
print("\nzero fraction:")
print((features[cols] == 0).mean().round(4))
print((features[["g_in_degree", "g_out_degree"]] == 0).mean().round(4))

        g_clustering  g_two_hop_out  g_in_weighted  g_out_weighted  \
count  513284.000000  513284.000000   5.132840e+05    5.132840e+05   
mean        0.017427      23.660611   4.565978e+07    4.565978e+07   
std         0.112952     517.001263   3.796664e+09    4.963903e+09   
min         0.000000       0.000000   0.000000e+00    0.000000e+00   
25%         0.000000       0.000000   1.267050e+02    4.758675e+02   
50%         0.000000       0.000000   1.230234e+04    8.657125e+03   
75%         0.000000       1.000000   2.026542e+05    1.383696e+05   
max         1.000000   16788.000000   1.046509e+12    2.026048e+12   

         g_net_flow  
count  5.132840e+05  
mean  -1.337750e-10  
std    4.379503e+09  
min   -1.046302e+12  
25%   -1.930763e+03  
50%    0.000000e+00  
75%    1.168760e+04  
max    1.046509e+12  

zero fraction:
g_clustering      0.9571
g_two_hop_out     0.6922
g_in_weighted     0.1819
g_out_weighted    0.0380
g_net_flow        0.1998
dtype: float64
g_in_degree    

In [10]:
import gc

df = pd.read_parquet(DATA_PROC / "01_audit.parquet")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
print(f"transactions: {len(df):,}")

# Rebuild the 70/15/15 temporal split from quantiles — same route as Notebooks 04 and 05.
# Never join 03_split_index.parquet: it has no key back to these rows.
t70 = df["Timestamp"].quantile(0.70)
t85 = df["Timestamp"].quantile(0.85)
df["split"] = np.where(df["Timestamp"] <= t70, "train",
              np.where(df["Timestamp"] <= t85, "val", "test"))

counts = df["split"].value_counts().to_dict()
print(counts)
assert counts == {"train": 3_554_957, "val": 761_749, "test": 761_639}, "split did not reproduce"

SENDER_COL, RECEIVER_COL = "Account", "Account.1"
df[SENDER_COL]   = df[SENDER_COL].astype(str)
df[RECEIVER_COL] = df[RECEIVER_COL].astype(str)

# Merge on account ID, not on row position. account_id is unique, so row count cannot change.
n_before = len(df)
df = df.merge(features.add_prefix("from_").rename(columns={"from_account_id": SENDER_COL}),
              on=SENDER_COL, how="left")
df = df.merge(features.add_prefix("to_").rename(columns={"to_account_id": RECEIVER_COL}),
              on=RECEIVER_COL, how="left")
assert len(df) == n_before, "merge changed the row count"

graph_cols = [f"{p}g_{c}" for p in ("from_", "to_")
              for c in ("pagerank","in_degree","out_degree","in_weighted",
                        "out_weighted","clustering","net_flow","two_hop_out")]
print(f"\ngraph columns attached: {len(graph_cols)}  (expect 16)")

# Missing values by split. Train must be ZERO: the graph was built from train edges,
# so every train account is already a vertex. Gaps can only be val/test newcomers.
miss = df.groupby("split")[["from_g_pagerank", "to_g_pagerank"]].apply(lambda s: s.isna().sum())
print("\nmissing by split:\n", miss)
assert df.loc[df["split"] == "train", graph_cols].isna().sum().sum() == 0, \
    "training rows have missing graph features — graph/audit mismatch"

# Integrity check (iii): how many accounts appear only after the training window.
seen = set(features["account_id"])
all_accts = set(df[SENDER_COL]) | set(df[RECEIVER_COL])
unseen = all_accts - seen
print(f"\nintegrity check (iii) — accounts total {len(all_accts):,}  (expect 515,080)")
print(f"  in training graph : {len(seen):,}  (expect 513,284)")
print(f"  unseen (post-train): {len(unseen):,}  (expect 1,796 = 0.35%)")

del g, g_simple; gc.collect()

transactions: 5,078,345
{'train': 3554957, 'val': 761749, 'test': 761639}

graph columns attached: 16  (expect 16)

missing by split:
        from_g_pagerank  to_g_pagerank
split                                
test               950            707
train                0              0
val                761            612

integrity check (iii) — accounts total 515,080  (expect 515,080)
  in training graph : 513,284  (expect 513,284)
  unseen (post-train): 1,796  (expect 1,796 = 0.35%)


0

In [11]:
COUNT_FEATS = ["g_in_degree", "g_out_degree", "g_two_hop_out"]
NUM_FEATS   = ["g_pagerank", "g_in_weighted", "g_out_weighted", "g_clustering", "g_net_flow"]
LABEL = "Is Laundering"

# Decision (b), 26 Aug: median across training ACCOUNTS, each counted once.
# `features` already contains exactly the 513,284 training-graph accounts.
acct_median = {f: float(features[f].median()) for f in NUM_FEATS}

# The rejected alternative, measured not assumed — so the report can state the gap.
train = df["split"] == "train"
txn_median = {f: float(pd.concat([df.loc[train, f"from_{f}"], df.loc[train, f"to_{f}"]]).median())
              for f in NUM_FEATS}

print(f"{'feature':<16}{'account basis':>18}{'transaction basis':>20}{'ratio':>10}")
for f in NUM_FEATS:
    a, t = acct_median[f], txn_median[f]
    r = (t / a) if a not in (0.0,) else float("nan")
    print(f"{f:<16}{a:>18.6g}{t:>20.6g}{r:>10.2f}")

# One median per feature, applied to both sender and receiver: with an account-level
# basis there is a single population, so from_ and to_ share the value.
for f in NUM_FEATS:
    for p in ("from_", "to_"):
        df[f"{p}{f}"] = df[f"{p}{f}"].fillna(acct_median[f])
for f in COUNT_FEATS:
    for p in ("from_", "to_"):
        df[f"{p}{f}"] = df[f"{p}{f}"].fillna(0)

assert df[graph_cols].isna().sum().sum() == 0, "NaNs survived imputation"
print("\nzero NaN across all 16 graph columns")

# Integrity check (ii): a post-training newcomer must carry EXACTLY the imputed values.
probe = next(iter(unseen))
row = df[(df[SENDER_COL] == probe) | (df[RECEIVER_COL] == probe)].iloc[0]
side = "from_" if row[SENDER_COL] == probe else "to_"
print(f"\nintegrity check (ii) — unseen account {probe}, side {side}, split {row['split']}")
for f in NUM_FEATS:
    print(f"  {side}{f:<16} {row[side+f]:>16.6g}   imputed {acct_median[f]:>16.6g}   "
          f"{'MATCH' if row[side+f] == acct_median[f] else 'MISMATCH'}")
for f in COUNT_FEATS:
    print(f"  {side}{f:<16} {row[side+f]:>16.6g}   imputed {0:>16}   "
          f"{'MATCH' if row[side+f] == 0 else 'MISMATCH'}")

# Staleness strata on test-main (wind-down = 11-18 Sep, excluded).
test_main = (df["split"] == "test") & (df["Timestamp"] < pd.Timestamp("2022-09-11"))
s_unseen = df.loc[test_main, SENDER_COL].isin(unseen)
r_unseen = df.loc[test_main, RECEIVER_COL].isin(unseen)
n_imp = s_unseen.astype(int) + r_unseen.astype(int)
print("\nstaleness strata on test-main (expect 758,982 / 1,491 / 58):")
for k, name in [(0, "both accounts active"), (1, "one imputed"), (2, "both imputed")]:
    sel = n_imp == k
    print(f"  {name:<22} n={int(sel.sum()):>8,}   illicit={int(df.loc[test_main][LABEL][sel].sum()):>5,}")

with open(DATA_PROC / "05_imputation_values.json", "w") as fh:
    json.dump({"basis": "median across training accounts, each counted once (decision b, 2026-08-26)",
               "account_basis_applied": acct_median,
               "transaction_basis_rejected": txn_median,
               "count_features_filled_with": 0,
               "rows_affected": int(3030)}, fh, indent=2)

df.to_parquet(DATA_PROC / "05_features_full.parquet", compression="zstd")
print(f"\nsaved 05_features_full.parquet — {len(df):,} rows, {df.shape[1]} columns")

feature              account basis   transaction basis     ratio
g_pagerank             1.34466e-06         9.98489e-07      0.74
g_in_weighted              12302.3              180635     14.68
g_out_weighted             8657.12              170169     19.66
g_clustering                     0                   0       nan
g_net_flow                       0            0.922572       nan

zero NaN across all 16 graph columns

integrity check (ii) — unseen account 8140891F0, side from_, split val
  from_g_pagerank            1.34466e-06   imputed      1.34466e-06   MATCH
  from_g_in_weighted             12302.3   imputed          12302.3   MATCH
  from_g_out_weighted            8657.12   imputed          8657.12   MATCH
  from_g_clustering                    0   imputed                0   MATCH
  from_g_net_flow                      0   imputed                0   MATCH
  from_g_in_degree                     0   imputed                0   MATCH
  from_g_out_degree                    0   i